KAGGLE NLP COURSE
* Cleaning Text Data (1)


In [34]:
# Import des librairies
import numpy as np
import pandas as pd
import re
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /home/marina/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [20]:
#charger le csv
#toujours checker voir l'encodage sinon kpoaaa egble
df = pd.read_csv("./data/spam.csv", encoding = 'latin-1')
df.head()
print(df.shape)

(5572, 5)


In [21]:
#avant de passer au nettoyage initialiser le dataset lieke v1 et vw on peut pas se retrouver alors on va mettre en de vrais noms
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

#on va verifier voir si les wes la on disparus finalement
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [22]:
#on va maintenanr verifier tous les trucs qui sony nuls avec le keyword(qui est une fonction btw) isnull()
df.isnull().sum()

label      0
message    0
dtype: int64

In [23]:
#checker les trucs qui se dupliquent
df.duplicated().sum()

np.int64(403)

In [24]:
#on va maintenant supprimer tous ceux qui se dupliquent
df.drop_duplicates(keep='first')
#le keyword keep = first c'est pout sire de supprimer le premier qui se repeat et de laisser l'autre

print('----------------------------------------')
print(df.shape)
print('----------------------------------------')

print(df['label'].value_counts())

----------------------------------------
(5572, 2)
----------------------------------------
label
ham     4825
spam     747
Name: count, dtype: int64


* on peut voir que juste apres ses petites manipualtions notre shapes passe de 5 a 2 en mode on a supprier tout ce qui est supperflu


In [26]:
#changer soit en upper ou lower case
df['message'] = df['message'].str.lower()

df.head()

,label,message
0,ham,"go until jurong point, crazy.. available only ..."
1,ham,ok lar... joking wif u oni...
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor... u c already then say...
4,ham,"nah i don't think he goes to usf, he lives aro..."


In [28]:
# on va maintenant enlever les ponctuations avce la librairie re en py ofc
df['message'] = df['message'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

df.head()

,label,message
0,ham,go until jurong point crazy available only in ...
1,ham,ok lar joking wif u oni
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor u c already then say
4,ham,nah i dont think he goes to usf he lives aroun...


In [35]:
# on maintenanr passer a la tokenisation
df['tokens'] = df['message'].apply(word_tokenize)

print(df[['message', 'tokens']].head())

                                             message  \
0  go until jurong point crazy available only in ...   
1                            ok lar joking wif u oni   
2  free entry in 2 a wkly comp to win fa cup fina...   
3        u dun say so early hor u c already then say   
4  nah i dont think he goes to usf he lives aroun...   

                                              tokens  
0  [go, until, jurong, point, crazy, available, o...  
1                     [ok, lar, joking, wif, u, oni]  
2  [free, entry, in, 2, a, wkly, comp, to, win, f...  
3  [u, dun, say, so, early, hor, u, c, already, t...  
4  [nah, i, dont, think, he, goes, to, usf, he, l...  


* On peut tout naturellement voir comment apres la tokenisation nos phrases ont comme mnt des listes wouhh


In [36]:
#on va maintenant passer a la suppression des stopwords deh comme nous sommes en anglais ce sont des most come the, is, in
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# Apply a filter to keep only the tokens that are NOT in our stop_words list
df['tokens'] = df['tokens'].apply(lambda x: [word for word in x if word not in stop_words])

print(df['tokens'].head())

[nltk_data] Downloading package stopwords to /home/marina/nltk_data...


0    [go, jurong, point, crazy, available, bugis, n...
1                       [ok, lar, joking, wif, u, oni]
2    [free, entry, 2, wkly, comp, win, fa, cup, fin...
3        [u, dun, say, early, hor, u, c, already, say]
4    [nah, dont, think, goes, usf, lives, around, t...
Name: tokens, dtype: object


[nltk_data]   Unzipping corpora/stopwords.zip.


* On peut dirct voir que des motes comme in, he, et autre ne sont plus dans nos listes

# Stemming vs. Lemmatization
En NLP (Natural Language Processing), Stemming et Lemmatization sont deux techniques qui permettent de ramener les mots à une forme plus simple pour faciliter leur analyse.
* Le Stemming coupe les mots
* La Lemmatization cherche leur forme de base réelle.

In [37]:
from nltk.stem import PorterStemmer

ps = PorterStemmer()
df['tokens'] = df['tokens'].apply(
    lambda words: [ps.stem(word) for word in words]
)

df.head()

,label,message,tokens
0,ham,go until jurong point crazy available only in ...,"[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,ok lar joking wif u oni,"[ok, lar, joke, wif, u, oni]"
2,spam,free entry in 2 a wkly comp to win fa cup fina...,"[free, entri, 2, wkli, comp, win, fa, cup, fin..."
3,ham,u dun say so early hor u c already then say,"[u, dun, say, earli, hor, u, c, alreadi, say]"
4,ham,nah i dont think he goes to usf he lives aroun...,"[nah, dont, think, goe, usf, live, around, tho..."


In [38]:
print(df.shape)

(5572, 3)


In [39]:
#joindre les tokens back into text
df['processed_message'] = df['tokens'].apply(lambda x: " ".join(x))
df[['tokens','processed_message']].head()

,tokens,processed_message
0,"[go, jurong, point, crazi, avail, bugi, n, gre...",go jurong point crazi avail bugi n great world...
1,"[ok, lar, joke, wif, u, oni]",ok lar joke wif u oni
2,"[free, entri, 2, wkli, comp, win, fa, cup, fin...",free entri 2 wkli comp win fa cup final tkt 21...
3,"[u, dun, say, earli, hor, u, c, alreadi, say]",u dun say earli hor u c alreadi say
4,"[nah, dont, think, goe, usf, live, around, tho...",nah dont think goe usf live around though


In [40]:
#maintenant save nos modifis
df.to_csv('cleaned_spam.csv', index=False)
print("Cleaned data saved as 'cleaned_spam.csv'")

Cleaned data saved as 'cleaned_spam.csv'


### Nous avons fini la premiere partie bien evidemment mais on a un problem majeur qui : UN MODEL DE ML NE PEUT PAS LIRE CE CHARABIA
#### Donc on continue avec les extractions dans le notebook feature_extraction